In [4]:
from google.cloud import bigquery
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

client = bigquery.Client(project="YOUR_PROJECT_ID")

ModuleNotFoundError: No module named 'sklearn'

In [5]:
from google.cloud import bigquery

client = bigquery.Client(project="modelraja")

query = """
SELECT * FROM `modelraja.MODELS_DATASET.VW_top_gainers_training_data` 
"""

df = client.query(query).to_dataframe(create_bqstorage_client=True)
print(df.head())

  Ticker  Stock_Price Stock_Snapshot_Date Performance_Week  \
0    CCO         2.24          2026-01-28            7.69%   
1   JBLU         5.04          2026-01-28           -8.70%   
2    PFS        22.22          2026-01-28            4.86%   
3    XRX         2.33          2026-01-28            3.56%   
4   VSAT        47.58          2026-01-28           11.59%   

  next_day_Snapshot_Date  eod_next_High  nextday_Low         Earnings_Date  \
0             2026-01-29           2.30         2.22  11/6/2025 8:30:00 AM   
1             2026-01-29           5.30         4.97  1/27/2026 8:30:00 AM   
2             2026-01-29          22.37        21.70  1/27/2026 4:30:00 PM   
3             2026-01-29           2.29         2.03  1/29/2026 8:30:00 AM   
4             2026-01-29          49.51        45.18  11/7/2025 4:30:00 PM   

   Average_True_Range  Change  ... price_value_change_nextday_day_high  \
0                0.10  10.34%  ...                                 0.1   
1         

In [6]:
df.head()

,Ticker,Stock_Price,Stock_Snapshot_Date,Performance_Week,next_day_Snapshot_Date,eod_next_High,nextday_Low,Earnings_Date,Average_True_Range,Change,...,price_value_change_nextday_day_high,price_value_change_nextday_day_low,nextday_range,price_value_delta_above_put_flag,price_value_delta_above_call_flag,strike_to_close_price_gap,days_to_earnings,todays_range,days_earnings_to_expiry,days_to_options_expiry
0,CCO,2.24,2026-01-28,7.69%,2026-01-29,2.30,2.22,11/6/2025 8:30:00 AM,0.10,10.34%,...,0.1,0.0,0.1,Beat,NoBeat,0.24,-83,0.25,106.0,23.0
1,JBLU,5.04,2026-01-28,-8.70%,2026-01-29,5.30,4.97,1/27/2026 8:30:00 AM,0.28,6.55%,...,0.3,0.1,0.3,Beat,NoBeat,0.04,-1,0.41,3.0,2.0
2,PFS,22.22,2026-01-28,4.86%,2026-01-29,22.37,21.70,1/27/2026 4:30:00 PM,0.59,6.67%,...,0.2,0.5,0.7,NoBeat,Beat,-0.28,-1,1.22,24.0,23.0
3,XRX,2.33,2026-01-28,3.56%,2026-01-29,2.29,2.03,1/29/2026 8:30:00 AM,0.16,9.39%,...,0.0,0.3,0.3,NoBeat,NoBeat,0.33,1,0.31,22.0,23.0
4,VSAT,47.58,2026-01-28,11.59%,2026-01-29,49.51,45.18,11/7/2025 4:30:00 PM,2.54,5.64%,...,1.9,2.4,4.3,NoBeat,NoBeat,-0.42,-82,3.31,105.0,23.0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4798 entries, 0 to 4797
Data columns (total 26 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Ticker                               4798 non-null   object 
 1   Stock_Price                          4798 non-null   float64
 2   Stock_Snapshot_Date                  4798 non-null   object 
 3   Performance_Week                     4798 non-null   object 
 4   next_day_Snapshot_Date               4798 non-null   object 
 5   eod_next_High                        4724 non-null   float64
 6   nextday_Low                          4724 non-null   float64
 7   Earnings_Date                        4794 non-null   object 
 8   Average_True_Range                   4798 non-null   float64
 9   Change                               4798 non-null   object 
 10  Call_Option_Expiry_Date              4798 non-null   object 
 11  Call_Option_Strike            

In [8]:
df.columns

Index(['Ticker', 'Stock_Price', 'Stock_Snapshot_Date', 'Performance_Week',
       'next_day_Snapshot_Date', 'eod_next_High', 'nextday_Low',
       'Earnings_Date', 'Average_True_Range', 'Change',
       'Call_Option_Expiry_Date', 'Call_Option_Strike', 'Put_Option_Strike',
       'Call_Option_Price', 'Put_Option_Expiry_Date', 'Put_Option_Price',
       'price_value_change_nextday_day_high',
       'price_value_change_nextday_day_low', 'nextday_range',
       'price_value_delta_above_put_flag', 'price_value_delta_above_call_flag',
       'strike_to_close_price_gap', 'days_to_earnings', 'todays_range',
       'days_earnings_to_expiry', 'days_to_options_expiry'],
      dtype='object')

In [9]:
# Drop specified columns from existing dataframe `df`
cols_to_drop = ["Put_Option_Expiry_Date", "Call_Option_Expiry_Date",
                 "price_value_delta_above_put_flag","next_day_Snapshot_Date","Earnings_Date","nextday_range"
                 ,'price_value_change_nextday_day_high', 'price_value_change_nextday_day_low', 'eod_nextday_High', 'eod_nextday_Low',"Put_Option_Strike"]
found = [c for c in cols_to_drop if c in df.columns]
if found:
    df.drop(columns=found, inplace=True)
print("Dropped columns:", found)
print("Remaining columns:", df.columns.tolist())


Dropped columns: ['Put_Option_Expiry_Date', 'Call_Option_Expiry_Date', 'price_value_delta_above_put_flag', 'next_day_Snapshot_Date', 'Earnings_Date', 'nextday_range', 'price_value_change_nextday_day_high', 'price_value_change_nextday_day_low', 'eod_next_High', 'nextday_Low', 'Put_Option_Strike']
Remaining columns: ['Ticker', 'Stock_Price', 'Stock_Snapshot_Date', 'Performance_Week', 'Average_True_Range', 'Change', 'Call_Option_Strike', 'Call_Option_Price', 'Put_Option_Price', 'price_value_delta_above_call_flag', 'strike_to_close_price_gap', 'days_to_earnings', 'todays_range', 'days_earnings_to_expiry', 'days_to_options_expiry']


In [10]:
df.columns

Index(['Ticker', 'Stock_Price', 'Stock_Snapshot_Date', 'Performance_Week',
       'Average_True_Range', 'Change', 'Call_Option_Strike',
       'Call_Option_Price', 'Put_Option_Price',
       'price_value_delta_above_call_flag', 'strike_to_close_price_gap',
       'days_to_earnings', 'todays_range', 'days_earnings_to_expiry',
       'days_to_options_expiry'],
      dtype='object')

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

CALL_TARGET_COL = "price_value_delta_above_call_flag"
PUT_TARGET_COL = "price_value_delta_above_put_flag"
ID_COLS = ["Ticker", "Stock_Snapshot_Date"]

working_df = df.copy()
print("Rows:", len(working_df))
print("Columns:", working_df.columns.tolist())



In [ ]:
def train_target_model(input_df: pd.DataFrame, target_col: str, model_tag: str):
    print(f"\n===== Training target: {target_col} ({model_tag}) =====")

    drop_cols = [CALL_TARGET_COL, PUT_TARGET_COL] + ID_COLS
    feature_cols = [c for c in input_df.columns if c not in drop_cols]

    train_df = input_df[input_df[target_col].notna()].copy()
    if train_df.empty:
        raise ValueError(f"No non-null rows found for target: {target_col}")

    X = train_df[feature_cols].copy()
    y = train_df[target_col].copy()
    id_df = train_df[ID_COLS].copy() if all(c in train_df.columns for c in ID_COLS) else pd.DataFrame(index=train_df.index)

    numeric_features = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric_features),
            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_features),
        ]
    )

    X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
        X, y, id_df, test_size=0.2, random_state=42, stratify=y
    )

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc = le.transform(y_test)

    xgb_clf = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    model_xgb = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", xgb_clf),
    ])

    model_xgb.fit(X_train, y_train_enc)

    pred_base_enc = model_xgb.predict(X_test)
    pred_base = le.inverse_transform(pred_base_enc)
    y_test_labels = le.inverse_transform(y_test_enc)

    print("Base Accuracy:", accuracy_score(y_test_labels, pred_base))
    print("Base Classification Report:\n", classification_report(y_test_labels, pred_base))
    print("Base Confusion Matrix:\n", confusion_matrix(y_test_labels, pred_base))

    param_dist = {
        "classifier__n_estimators": [100, 200, 300, 500],
        "classifier__max_depth": [3, 5, 6, 8],
        "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "classifier__subsample": [0.6, 0.8, 1.0],
        "classifier__colsample_bytree": [0.6, 0.8, 1.0],
        "classifier__gamma": [0, 1, 5],
    }

    search = RandomizedSearchCV(
        estimator=model_xgb,
        param_distributions=param_dist,
        n_iter=20,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )

    search.fit(X_train, y_train_enc)
    best_model = search.best_estimator_

    pred_tuned_enc = best_model.predict(X_test)
    pred_tuned = le.inverse_transform(pred_tuned_enc)

    print("Best params:", search.best_params_)
    print("Best CV score:", search.best_score_)
    print("Tuned Accuracy:", accuracy_score(y_test_labels, pred_tuned))
    print("Tuned Classification Report:\n", classification_report(y_test_labels, pred_tuned))
    print("Tuned Confusion Matrix:\n", confusion_matrix(y_test_labels, pred_tuned))

    out_df = id_test.copy()
    out_df["target_type"] = model_tag
    out_df["actual"] = y_test_labels
    out_df["predicted"] = pred_tuned

    if hasattr(best_model.named_steps["classifier"], "predict_proba"):
        pred_proba = best_model.predict_proba(X_test)
        for i, cls in enumerate(le.classes_):
            out_df[f"prob_{cls}"] = pred_proba[:, i]

    pred_path = f"xgboost_tuned_predictions_{model_tag}.csv"
    out_df.to_csv(pred_path, index=False)
    print(f"Saved tuned predictions to {pred_path}")

    return {
        "best_model": best_model,
        "model_xgb": model_xgb,
        "label_encoder": le,
    }


call_artifacts = train_target_model(working_df, CALL_TARGET_COL, "call")
put_artifacts = train_target_model(working_df, PUT_TARGET_COL, "put")

# Backward-compatibility variable names
best_model = call_artifacts["best_model"]
model_xgb = call_artifacts["model_xgb"]
le = call_artifacts["label_encoder"]



In [ ]:
import joblib
from pathlib import Path

out_dir = Path("models")
out_dir.mkdir(exist_ok=True)

artifact_map = {
    "best_model.joblib": call_artifacts["best_model"],
    "model_xgb.joblib": call_artifacts["model_xgb"],
    "label_encoder.joblib": call_artifacts["label_encoder"],
    "best_model_put.joblib": put_artifacts["best_model"],
    "model_xgb_put.joblib": put_artifacts["model_xgb"],
    "label_encoder_put.joblib": put_artifacts["label_encoder"],
}

# Optional: keep logistic model if it exists from old cells
legacy_model = globals().get("model")
if legacy_model is not None:
    artifact_map["model_logreg.joblib"] = legacy_model

saved = []
for fname, obj in artifact_map.items():
    p = out_dir / fname
    joblib.dump(obj, p)
    saved.append(str(p))

print("Saved files:", saved)

